# TLGMapper Compliance Validation Notebook

This notebook provides interactive compliance validation for the TLGMapper repository, allowing you to verify governance, provenance, and structural integrity with rich output and step-by-step analysis.

In [ ]:
# Import required libraries
import os
import sys
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, HTML

In [ ]:
# Configuration
REPO_ROOT = Path.cwd()
REQUIRED_FILES = [
    'ASSET_PROVENANCE.md',
    'SECURITY.md', 
    'DEPENDENCIES.md',
    '.github/CODEOWNERS',
    '.github/workflows/ci.yml',
    '.gitignore',
    'README.md',
    'TLGMapper.py'
]

## Governance File Presence Check

Verify that all required compliance and governance files are present in the repository.

In [ ]:
def check_governance_files():
    results = []
    for file_path in REQUIRED_FILES:
        exists = (REPO_ROOT / file_path).exists()
        results.append({
            'File': file_path,
            'Status': '✅ Present' if exists else '❌ Missing',
            'Exists': exists
        })
    
    df = pd.DataFrame(results)
    display(df)
    
    missing_count = len([r for r in results if not r['Exists']])
    if missing_count == 0:
        display(Markdown("**🎉 All governance files are present!**"))
    else:
        display(Markdown(f"**⚠️ {missing_count} governance files are missing.**"))
    
    return results

governance_results = check_governance_files()

## Provider Index Validation

Validate the structural integrity of TraceLogging provider index files.

In [ ]:
def validate_provider_indexes():
    provider_dir = REPO_ROOT / 'TraceLoggingProviders'
    
    if not provider_dir.exists():
        display(Markdown("**❌ TraceLoggingProviders directory not found**"))
        return []
    
    json_files = list(provider_dir.glob('tlg_provider_index_*.json'))
    
    if not json_files:
        display(Markdown("**❌ No provider index JSON files found**"))
        return []
    
    results = []
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Basic validation
            is_valid = isinstance(data, dict)
            provider_count = len(data.get('providers', [])) if is_valid else 0
            
            results.append({
                'File': json_file.name,
                'Valid JSON': is_valid,
                'Providers': provider_count,
                'Status': '✅ Valid' if is_valid else '❌ Invalid'
            })
            
        except Exception as e:
            results.append({
                'File': json_file.name,
                'Valid JSON': False,
                'Providers': 0,
                'Status': f'❌ Error: {str(e)}'
            })
    
    df = pd.DataFrame(results)
    display(df)
    
    valid_count = len([r for r in results if r['Valid JSON']])
    display(Markdown(f"**📊 Summary: {valid_count}/{len(results)} provider indexes are valid**"))
    
    return results

provider_results = validate_provider_indexes()

## Provenance Documentation Analysis

Analyze the asset provenance documentation for completeness and accuracy.

In [ ]:
def analyze_provenance():
    provenance_file = REPO_ROOT / 'ASSET_PROVENANCE.md'
    
    if not provenance_file.exists():
        display(Markdown("**❌ ASSET_PROVENANCE.md not found**"))
        return
    
    try:
        with open(provenance_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Extract sections
        sections = content.split('## ')
        provenance_sections = [s for s in sections if 'Provider Index' in s]
        
        results = []
        for section in provenance_sections:
            lines = section.split('\n')
            title = lines[0].strip()
            
            # Extract key information
            file_line = next((l for l in lines if l.startswith('**File:**')), '')
            build_line = next((l for l in lines if l.startswith('**Build Number:**')), '')
            
            file_path = file_line.replace('**File:**', '').strip() if file_line else ''
            build_num = build_line.replace('**Build Number:**', '').strip() if build_line else ''
            
            # Check if file exists
            file_exists = (REPO_ROOT / file_path).exists() if file_path else False
            
            results.append({
                'Dataset': title,
                'File Path': file_path,
                'Build Number': build_num,
                'File Exists': '✅ Yes' if file_exists else '❌ No'
            })
        
        df = pd.DataFrame(results)
        display(df)
        
        missing_files = len([r for r in results if '❌' in r['File Exists']])
        if missing_files == 0:
            display(Markdown("**✅ All provenance-documented files exist**"))
        else:
            display(Markdown(f"**⚠️ {missing_files} provenance-documented files are missing**"))
        
        return results
        
    except Exception as e:
        display(Markdown(f"**❌ Error analyzing provenance: {str(e)}**"))
        return []

provenance_results = analyze_provenance()

## Compliance Status Report

Generate a comprehensive compliance status report.

In [ ]:
def generate_compliance_report():
    report = []
    
    # Governance files
    gov_passed = all(r['Exists'] for r in governance_results)
    report.append({
        'Domain': 'Governance Files',
        'Status': '✅ Complete' if gov_passed else '❌ Incomplete',
        'Details': f"{len([r for r in governance_results if r['Exists']])}/{len(REQUIRED_FILES)} files present"
    })
    
    # Provider indexes
    if provider_results:
        prov_valid = all(r['Valid JSON'] for r in provider_results)
        report.append({
            'Domain': 'Provider Indexes',
            'Status': '✅ Valid' if prov_valid else '❌ Invalid',
            'Details': f"{len([r for r in provider_results if r['Valid JSON']])}/{len(provider_results)} files valid"
        })
    else:
        report.append({
            'Domain': 'Provider Indexes',
            'Status': '❌ Missing',
            'Details': 'No provider index files found'
        })
    
    # Provenance
    if provenance_results:
        prov_files_exist = all('✅' in r['File Exists'] for r in provenance_results)
        report.append({
            'Domain': 'Provenance Accuracy',
            'Status': '✅ Accurate' if prov_files_exist else '❌ Inaccurate',
            'Details': f"{len([r for r in provenance_results if '✅' in r['File Exists']])}/{len(provenance_results)} files verified"
        })
    else:
        report.append({
            'Domain': 'Provenance Accuracy',
            'Status': '❌ Missing',
            'Details': 'Provenance analysis failed'
        })
    
    df = pd.DataFrame(report)
    display(df)
    
    overall_status = all('✅' in r['Status'] for r in report)
    if overall_status:
        display(Markdown("## 🎉 Compliance Status: PASSED"))
        display(Markdown("**All compliance checks have passed. The repository is ready for enterprise use.**"))
    else:
        display(Markdown("## ⚠️ Compliance Status: NEEDS ATTENTION"))
        display(Markdown("**Some compliance checks failed. Please review and address the issues above.**"))
    
    return report

final_report = generate_compliance_report()

## Next Steps

If any compliance checks failed:

1. **Missing Files**: Create the missing governance files following the templates in the PR description
2. **Invalid JSON**: Validate and repair provider index JSON files
3. **Provenance Issues**: Ensure all documented files exist and update documentation as needed

Run this notebook again after making changes to verify compliance.